In [0]:
max_date = spark.sql("""
SELECT MAX(claim_date)
FROM main.healthcare_project_pipe.silver_claims
""").collect()[0][0]

In [0]:
print(max_date)

In [0]:
df_claims = spark.table("main.healthcare_project_pipe.silver_claims")

In [0]:
from pyspark.sql.functions import col, lit

new_claims = df_claims.filter(
    col("claim_date") > lit(max_date)
)

display(new_claims)
#new_claims.write.mode("append").saveAsTable("main.healthcare_project_pipe.claims")

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(
    spark, "main.healthcare_project_pipe.silver_claims"
)

delta_table.alias("t").merge(
    new_claims.alias("s"),
    "t.claim_id = s.claim_id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

In [0]:
%sql
SELECT * FROM main.healthcare_project_pipe.silver_claims

In [0]:
df_claims.filter(
    col("claim_id").isNull()
).count()

In [0]:
df_claims.groupBy("claim_id") \
    .count() \
    .filter("count > 1")

In [0]:
df_claims.filter(
    col("claim_amount") < 0
)

In [0]:
bronze_count = spark.table("main.healthcare_project_pipe.silver_claims").count()

silver_count = spark.table("main.healthcare_project_pipe.silver_claims").count()

In [0]:
print(bronze_count)
print(silver_count)